<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
# GPT-2

Generación de texto simple con GPT-2

Importamos lo necesario

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

Cargamos el modelo, su tokenizer y el head.

In [ ]:
model_name = "gpt2" #Carga GPT2
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

GPT2Tokenizer:
- Usa Byte Pair Encoding (BPE) para tokenización
- Convierte texto en tokens que el modelo puede entender
- Maneja subpalabras para vocabulario eficiente
- Vocabulario de ~50,000 tokens

GPT2LMHeadModel:
- Modelo Transformer decoder-only para generación de texto
- LM Head = Language Modeling Head (capa final para predicción)
- Arquitectura autorregresiva: predice la siguiente palabra
- Usa self-attention para entender contexto
- Pre-entrenado en gran corpus de texto de internet

Realizamos un prompt de entrada

In [ ]:
prompt = "In the future, artificial intelligence will"
inputs = tokenizer(prompt, return_tensors="pt")

Generamos texto

In [ ]:
outputs = model.generate(
    inputs["input_ids"],
    max_length=50,
    num_return_sequences=1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Usamos pipeline para simplificar

Importamos pipeline de transformers

In [ ]:
from transformers import pipeline

Usamos la tarea de generación de texto con el modelo gpt-2

In [ ]:
generator = pipeline("text-generation", model="gpt2")

Generamos texto

In [ ]:
result = generator(
    "Machine learning will revolutionize",
    max_length=40,
    num_return_sequences=1
)
print(result[0]['generated_text'])

Probabilidades de la siguiente palabra

Realizamos un nuevo prompt.

In [ ]:
text = "Deep learning is"
inputs = tokenizer(text, return_tensors="pt")

Realizamos el forward pass.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

Obtenemos el top-5 de salidas

In [ ]:
last_token_logits = logits[0, -1, :]
probs = torch.softmax(last_token_logits, dim=-1)
topk = torch.topk(probs, 5)

In [ ]:
print("Top 5 predicciones:")
for idx, score in zip(topk.indices, topk.values):
    print(f"{tokenizer.decode([idx])} -> {score.item():.4f}")